In [6]:
from dotenv import load_dotenv
load_dotenv()


from google import genai
import os

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [7]:
from google import genai
print("Success")

Success


In [8]:
from google import genai

def llm(prompt):
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [9]:
llm('hey whats up')

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 21.497012184s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's great you discovered the course! To give you the most accurate answer, I need a little more information:

*   **What is the name of the course?**
*   **Where did you discover it (e.g., a specific university website, Coursera, edX, Udemy, a private platform, etc.)?**
*   **Did you see any mention of start dates, enrollment periods, or deadlines?**

Courses have different enrollment models:

*   **Self-paced courses** often allow you to join at any time.
*   **Cohort-based courses** have fixed start dates and usually a deadline to enroll before the cohort begins.
*   **University courses** follow academic calendars with specific registration periods.

Once you provide those details, I can tell you if you can join now or what your options are!


In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [ ]:
documents[500]

{'id': 'b5d37f88b6',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How are homework assignments scored?',
 'answer': "Homework grade = points for the questions + 1 point for the FAQ + up to 7 points for learning in public.\n\n- Learning in public: up to 7 points, depending on how many platforms you shared your post on (e.g. LinkedIn, X, blog) and the quality of the post.\n- FAQ: 1 point. To get it, contribute to the FAQ repo and add the link to your PR in your homework submission.\n\nOptional questions are scored too - read the submission form carefully. The homework is for practice and does not affect your certificate. If you think you weren't graded, log in to check your results or search for your name on the leaderboard."}

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
search_results = search(question)

In [ ]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [ ]:
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

NameError: name 'gemini_client' is not defined

In [ ]:
answer = response.text

In [ ]:
print(answer)

Yes, you can join the course now and start learning immediately. The videos and GitHub materials are available, and you can begin working through them whenever you want.

However, if you want to receive a **certificate**, there are specific conditions:
*   You need to submit your project while submissions are still being accepted.
*   You must finish the course with a "live" cohort, as this involves peer-reviewing 3 capstone projects after submitting your own. Peer reviews can only be done at the time the course is actively running, after the submission form is closed, and the peer-review list is compiled.

You can check the deadlines on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) to see if the project submission and peer-review windows are still open for the current cohort. If they have closed, you won't be able to get a certificate for this run, but you can still follow the course in a self-paced mode for learning.


In [10]:
response.usage_metadata

NameError: name 'response' is not defined

In [ ]:
from google.genai import types

response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS
    )
)

In [ ]:
from google.genai import types

def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    response = gemini_client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions
        )
    )

    return response.text

In [ ]:
answer = llm(
    instructions="You are a helpful assistant.",
    user_prompt="What is Python?"
)

print(answer)

**Python is a popular, high-level, interpreted, general-purpose programming language.**

It's known for its simplicity, readability, and versatility, making it an excellent choice for beginners and experienced developers alike.

Here's a breakdown of what that means and why it's so widely used:

1.  **High-Level:** You don't have to worry about low-level details like managing memory. Python handles much of that for you, allowing you to focus on solving the problem.
2.  **Interpreted:** Python code is executed line by line by an interpreter, rather than being compiled into machine code all at once. This makes the development process faster, as you can test changes immediately.
3.  **General-Purpose:** Python isn't specialized for one specific task. It can be used for a wide range of applications, from web development to scientific computing.
4.  **Readability:** Python's syntax is designed to be clear and concise, often resembling natural language. It uses indentation to define code blo

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(INSTRUCTIONS, user_prompt)

In [ ]:
from google.genai import types

def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    response = gemini_client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions
        )
    )

    return response.text

In [ ]:
answer = rag("What is Agentic RAG?")
print(answer)

In Agentic RAG, the model can decide when to call a tool or function. A tool/function is something the model can call when it needs external help, such as searching documents, querying a database, calling an API, or running a calculation.

This differs from basic RAG, where retrieval is usually a fixed step that happens before the model answers.


In [ ]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available.


: 

In [ ]:
print(type(answer))
print(len(answer))

<class 'str'>
346


In [ ]:
repr(answer)

"'In Agentic RAG, the model can decide when to call a tool or function. A tool/function is something the model can call when it needs external help, such as searching documents, querying a database, calling an API, or running a calculation.\\n\\nThis differs from basic RAG, where retrieval is usually a fixed step that happens before the model answers.'"

In [13]:
len(chunks)

NameError: name 'chunks' is not defined